In [82]:
# imports
import os 
import csv
import pandas as pd
import requests 
import io
import time # for processing climate API

### Dataset A - Bryophyte Observations
*Contains all of the data we actually used for our bryophytes.*

We kept two files (in DatasetAData):
* `T01A_Site_Physical_Characteristics_2692088856810991747` (T01A) 
* `T19B_Moss_Identification__since_2009__11290156903414781363` (T19B) 

Originally from `publicABMIData7878735501232611176` folder.

### After preprocessing

T01A should contain:
* ABMI Site - joining key
* Year - joining key
* Public Latitude
* Public Longitude - for spatial filtering to parkland/North Saskatchewan/Red Deer region

T19B should contain:
* ABMI Site - joining key
* Year - joining key 
* Scientific Name - for counting species richness and filtering invalid records
* Taxonomic Resolution - filter to only count records identified to species level
* Common name - for ease of manual checks

### Preprocess_A():

Reads T01A (site coordinates) and T19B (moss identification) from
the DatasetAData folder, filters and merges them, then saves
annual aggregated features to preprocessing_results/.

Usage:
    python preprocess_A.py

Invalid/missing value codes:
* VNA = Variable not applicable
* SNI = Species not identified
* SNR = Species needs review
* UID = Unable to identify
* DNC = Did not collect

In [83]:
# Constant for all 3 functions
OUT_DIR    = "preprocessing_results"

In [84]:
# Inputs used in data B function
DATA_DIR_A   = "DatasetAData"

T01A_FILE  = "T01A_Site_Physical_Characteristics_2692088856810991747.csv"
T19B_FILE  = "T19B_Moss_Identification__since_2009__11290156903414781363.csv"

# Invalid / missing-value codes used throughout ABMI raw data - from guide
ABMI_MISSING = {"SNI", "SNR", "DNC", "VNA", "NONE", "PNA", "UID", ""}

In [85]:
def preprocess_A():
    """
    Load, filter, merge, and aggregate the ABMI bryophyte dataset.

    Returns
    -------
    df_annual: pd.DataFrame
        One row per year with columns:
            Year, n_sites_sampled, species_richness,
            total_identifications, richness_per_site, ids_per_site
    """

    os.makedirs(OUT_DIR, exist_ok=True)

    # Load raw files
    t01a_raw = pd.read_csv(
        os.path.join(DATA_DIR_A, T01A_FILE),
        encoding="utf-8",
        low_memory=False,
        index_col=False,
    )
    t19b_raw = pd.read_csv(
        os.path.join(DATA_DIR_A, T19B_FILE),
        encoding="utf-8",
        low_memory=False,
        index_col=False,
    )

    print(f"T01A raw shape: {t01a_raw.shape}")
    print(f"T19B raw shape: {t19b_raw.shape}")

    # Strip whitespace from all column names (fixes trailing spaces in some headers)
    t01a_raw.columns = t01a_raw.columns.str.strip()
    t19b_raw.columns = t19b_raw.columns.str.strip()

    # We keep only the columns we need and drop duplicates for T01A
    t01a = (
        t01a_raw[["ABMI Site", "Year", "Public Latitude", "Public Longitude"]]
        .copy()
        .rename(columns={
            "ABMI Site": "abmi_site",
            "Year": "year",
            "Public Latitude": "latitude",
            "Public Longitude": "longitude",
        })
    )

    # Convert to numeric - coerce bad values to NaN
    t01a["latitude"] = pd.to_numeric(t01a["latitude"],  errors="coerce")
    t01a["longitude"] = pd.to_numeric(t01a["longitude"], errors="coerce")
    t01a["year"] = pd.to_numeric(t01a["year"],      errors="coerce")

    # Drop missing coordinates or year
    t01a = t01a.dropna(subset=["latitude", "longitude", "year"])

    # One coordinate record per site-year
    t01a = t01a.drop_duplicates(subset=["abmi_site", "year"])

    print(f"T01A after cleaning: {t01a.shape}")

    # We keep only the columns we need and drop invalid records for T19B
    t19b = (
        t19b_raw[["ABMI Site", "Year", "Scientific Name", "Taxonomic Resolution", "Common Name"]]
        .copy()
        .rename(columns={
            "ABMI Site": "abmi_site",
            "Year": "year",
            "Scientific Name": "scientific_name",
            "Taxonomic Resolution": "taxonomic_resolution",
            "Common Name": "common_name",
        })
    )

    t19b["year"] = pd.to_numeric(t19b["year"], errors="coerce")

    # Remove rows where scientific name is a missing value code
    t19b["scientific_name_clean"] = t19b["scientific_name"].astype(str).str.strip()
    t19b = t19b[~t19b["scientific_name_clean"].isin(ABMI_MISSING)]
  
    # Keep only species-level identifications for consistent richness counts

    t19b = t19b[t19b["taxonomic_resolution"].astype(str).str.strip().str.lower() == "species"]

    # Merge coordinates onto T19B records
    t19b_merged = t19b.merge(t01a, on=["abmi_site", "year"], how="left")

    # Flag rows that couldn't be matched to a coordinate
    unmatched = t19b_merged["latitude"].isna().sum()
    if unmatched > 0:
        print(f"WARNING: {unmatched} T19B rows have no matching T01A coordinate.")

    # Drop unmatched rows
    t19b_merged = t19b_merged.dropna(subset=["latitude", "longitude"])

    # Save the merged cleaned file for inspection
    merged_path = os.path.join(OUT_DIR, "merged_bryophyte_cleaned.csv")
    t19b_merged.to_csv(merged_path, index=False)
    print(f"\nSaved merged cleaned data: {merged_path}")

    # Computes bioindicator features per year

    # Sites sampled each year (denominator for per-site normalization)
    sites_per_year = (
        t19b_merged.groupby("year")["abmi_site"]
        .nunique()
        .rename("n_sites_sampled")
    )

    # Species richness: unique species across all sites in a given year
    richness_per_year = (
        t19b_merged.groupby("year")["scientific_name_clean"]
        .nunique()
        .rename("species_richness")
    )

    # Total valid identifications (proxy for bryophyte abundance / detectability)
    ids_per_year = (
        t19b_merged.groupby("year")["scientific_name_clean"]
        .count()
        .rename("total_identifications")
    )

    df_annual = (
        pd.concat([sites_per_year, richness_per_year, ids_per_year], axis=1)
        .reset_index()
        .rename(columns={"year": "Year"})
        .sort_values("Year")
    )

    # Effort-normalized metrics
    df_annual["richness_per_site"] = (
        df_annual["species_richness"] / df_annual["n_sites_sampled"]
    ).round(4)

    df_annual["ids_per_site"] = (
        df_annual["total_identifications"] / df_annual["n_sites_sampled"]
    ).round(4)

    # drop years with fewer than 10 sites sampled
    usable_years = df_annual[df_annual["n_sites_sampled"] >= 10]["Year"]

    df_annual = df_annual[df_annual["Year"].isin(usable_years)]

    # Save annual features
    annual_path = os.path.join(OUT_DIR, "bryophyte_annual_features.csv")
    df_annual.to_csv(annual_path, index=False)
    print(f"Saved annual features: {annual_path}")

    return df_annual

### Dataset B - Alberta Wheat Yield
*Contains the annual wheat yield data used as our target variable.*

We kept one file (in DatasetBData):
* `3210035901-eng.csv` (Statistics Canada Table 32-10-0359-01)

from: `https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3210035901`

Filters applied on download:
* Geography: Alberta only
* Type of crop: Wheat, all
* Statistic: Average yield (bushels per acre)
* Reference period: All available years (1908 to present)

### After preprocessing

The file should contain:
* Year - joining key
* wheat_yield_bu_acre - target variable for modeling

Two outputs are saved:
* `wheat_yield_full_historical.csv` - full 1908 to present series for EDA
* `wheat_yield_2010_2019.csv` - filtered to modeling window only

### preprocess_B():

Reads the Statistics Canada wheat yield CSV from the DatasetBData folder,
strips the 8-row metadata header and footer footnotes added by Statistics Canada,
cleans missing values, and saves both the full historical series
and the 2010-2019 modeling window to preprocessing_results folder.

Usage:
    python preprocess_B.py

Missing value codes:
* .. = Not available for a specific reference period (Statistics Canada convention)

In [86]:

# Inputs used in data B function
DATA_DIR_B = "DatasetBData"

YIELD_FILE = "3210035901-eng.csv"  

# Statistics Canada missing value code
STATCAN_MISSING = {".."}

In [87]:
def preprocess_B():
    """
    Load, clean, and filter the Statistics Canada Alberta wheat yield dataset.

    Returns
    -------
    df_model : pd.DataFrame
        One row per year for 2010-2019 with columns:
          Year, wheat_yield_bu_acre
    """

    os.makedirs(OUT_DIR, exist_ok=True)

   # Load raw file - skip 8 metadata rows at top and footer content at bottom
    df_raw = pd.read_csv(
        os.path.join(DATA_DIR_B, YIELD_FILE),
        encoding="utf-8",
        skiprows=8,
        skipfooter=37, # skips symbol legend, table corrections, footnotes, citation
        engine="python", # required when using skipfooter
        on_bad_lines="skip",
    )

    print(f"Raw shape: {df_raw.shape}")

    # Rename to standard column names - adjust left side if Stats Can header differs
    df = df_raw.rename(columns={
        df_raw.columns[0]: "Year",
        df_raw.columns[1]: "wheat_yield_bu_acre",
    })

    # Convert to numeric - coerce StatCan missing value ".." to NaN
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df["wheat_yield_bu_acre"] = pd.to_numeric(df["wheat_yield_bu_acre"], errors="coerce")

    # Drop missing values
    df = df.dropna(subset=["Year", "wheat_yield_bu_acre"])


    df["Year"] = df["Year"].astype(int)

    # Save full historical series for EDA
    eda_path = os.path.join(OUT_DIR, "wheat_yield_full_historical.csv")
    df.to_csv(eda_path, index=False)
    print(f"\nSaved full historical series: {eda_path}  ({len(df)} years)")

    # Filter to modeling window 2010-2019
    df_model = df[(df["Year"] >= 2010) & (df["Year"] <= 2019)].copy()

    # Save modeling window
    model_path = os.path.join(OUT_DIR, "wheat_yield_2010_2019.csv")
    df_model.to_csv(model_path, index=False)
    print(f"\nSaved modeling window: {model_path}")

    return df_model

In [88]:
def fetch_C(segments=STATION_SEGMENTS, out_dir=DATA_DIR_C):
    """
    Download daily ECCC climate CSVs for central Alberta (Red Deer).
    Two station IDs are needed to span 2010-2019 without gaps:
      - Station 2134  (RED DEER A)          covers 2010-2013
      - Station 51440 (RED DEER REGIONAL A) covers 2014-2019

    One file per year is saved to DatasetCData/.
    """
    os.makedirs(out_dir, exist_ok=True)

    # timeframe=2 = daily, timeframe=3 = monthly
    # Month/Day in URL are ignored for daily annual downloads but required by the endpoint
    base_url = (
        "https://climate.weather.gc.ca/climate_data/bulk_data_e.html"
        "?format=csv"
        "&stationID={sid}"
        "&Year={yr}"
        "&Month=1"
        "&Day=1"
        "&timeframe=2"
        "&submit=Download+Data"
    )

    for seg in segments:
        sid = seg["station_id"]
        for yr in seg["years"]:
            out_path = os.path.join(out_dir, f"eccc_daily_{yr}.csv")
            if os.path.exists(out_path):
                print(f"{yr}: already exists, skipping")
                continue

            url = base_url.format(sid=sid, yr=yr)
            r = requests.get(url, timeout=30)
            r.raise_for_status()

            with open(out_path, "wb") as f:
                f.write(r.content)
            print(f"{yr}:saved: {out_path}  ({len(r.content):,} bytes)")
            time.sleep(0.5)

    print(f"\nfetch_C complete. Files in {out_dir}/:")
    for fname in sorted(os.listdir(out_dir)):
        print(f"{fname}")

### Dataset C - Climate Variables (ECCC Daily Weather)
*Contains annual climate features derived from daily weather observations for central Alberta (Red Deer area).*

We kept ten files (in DatasetCData), one per year:
* `eccc_daily_2010.csv` through `eccc_daily_2019.csv`

Downloaded from: `https://climate.weather.gc.ca/climate_data/bulk_data_e.html`

Two station IDs are required to span 2010-2019 without gaps:
* Station 2134 (RED DEER A) — covers 2010-2013
* Station 51440 (RED DEER REGIONAL A) — covers 2014-2019

Parameters used on download:
* Format: CSV
* Timeframe: Daily (timeframe=2)
* One full year per request

### After preprocessing

The file should contain:
* Year - joining key
* mean_summer_temp_C - mean of daily Mean Temp for June, July, August
* total_growing_precip_mm - sum of daily Total Precip for May through August
* frost_free_days - count of days in May through September where Min Temp > 0 °C
* spring_temp_anomaly_C - April/May mean temp minus the 2010-2019 April/May mean

Two outputs are saved to the preprocessing_results folder:
* `climate_annual_full.csv` - full computed series across all loaded years, including `spring_mean_temp_C` for transparency
* `climate_2010_2019.csv` - filtered to the modeling window only, with `spring_mean_temp_C` dropped (anomaly column retained)

### preprocess_C():

Loads the ten ECCC daily CSVs from the DatasetCData folder,
concatenates them, and computes four annual climate features
used as predictors in the model. Saves both the full series
and the 2010-2019 modeling window to the preprocessing_results folder.

Usage:
    python preprocess_C.py

Missing value codes treated as NaN (ECCC conventions):
* M = Missing
* E = Estimated
* T = Trace (precipitation too small to measure)
* L = Precipitation may have occurred but was not measurable
* S = More than one observation in the hour

### Known limitations

* **Spring temperature anomaly is within-window only.** The anomaly is computed
  as deviation from the 2010-2019 mean rather than the standard 1981-2010 climate
  normal. This is flagged as a limitation — a true anomaly would require fetching
  the 30-year baseline period from the same stations.
* **Station discontinuity.** The switch from Station 2134 to Station 51440 in 2014
  introduces a potential instrumentation or siting discontinuity. Both stations
  are at Red Deer Regional Airport and the overlap is minor, but this is
  an acknowledged limitation.
* **2016 frost-free days anomaly.** The frost_free_days value for 2016 is 25,
  far below adjacent years (~134-148). This likely reflects missing minimum
  temperature records for that year rather than a true climate signal and
  should be flagged during EDA.
* **Missing precipitation 2018-2019.** total_growing_precip_mm is NaN for 2018
  and 2019, which will require imputation or exclusion during modeling.

In [89]:
# Inputs used in data C function
DATA_DIR_C = "DatasetCData"

CLIMATE_YEARS = range(2010, 2020)

# Two Red Deer stations needed to span 2010-2019 without a monthly gap
# Using daily (timeframe=2) since RED DEER A monthly only goes to 2007
STATION_SEGMENTS = [
    {"station_id": 2134,  "years": range(2010, 2014), "name": "RED_DEER_A"},
    {"station_id": 51440, "years": range(2014, 2020), "name": "RED_DEER_REGIONAL_A"},
]

In [90]:
def preprocess_C():
    """
    Load daily ECCC CSVs for Red Deer (two station segments covering 2010 to 2019),
    compute four annual climate features, and save outputs.

    Returns
    -------
    df_model : pd.DataFrame
        One row per year for 2010 to 2019 with columns:
          Year, mean_summer_temp_C, total_growing_precip_mm,
          frost_free_days, spring_temp_anomaly_C
    """
    os.makedirs(OUT_DIR, exist_ok=True)

    # Load all daily files
    frames = []
    for yr in CLIMATE_YEARS:
        fpath = os.path.join(DATA_DIR_C, f"eccc_daily_{yr}.csv")

        df_yr = pd.read_csv(
            fpath,
            encoding="utf-8-sig",
            na_values=["", "M", "E", "T", "L", "S", "<31"],
        )
        frames.append(df_yr)

    if not frames:
        raise RuntimeError("No climate files found. Run fetch_C() first.")

    df_raw = pd.concat(frames, ignore_index=True)
    print(f"\nCombined raw shape: {df_raw.shape}")

    # Rename to safe column names 
    df = df_raw.rename(columns={
        "Year":"year",
        "Month":"month",
        "Day":"day",
        "Mean Temp (\u00b0C)":"mean_temp", # \u00b0 is just for the degree symbol
        "Min Temp (\u00b0C)":"min_temp",
        "Total Precip (mm)":"total_precip",
    })

    # Convert to numeric 
    for col in ["year", "month", "day", "mean_temp", "min_temp", "total_precip"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["year"] = df["year"].astype("Int64")
    df["month"] = df["month"].astype("Int64")

    df = df.dropna(subset=["year", "month"])

    # Compute annual features 
    records = []

    for yr, grp in df.groupby("year"):
        summer = grp[grp["month"].isin([6, 7, 8])]
        growing = grp[grp["month"].isin([5, 6, 7, 8])]
        spring = grp[grp["month"].isin([4, 5])]
        ff_win = grp[grp["month"].isin([5, 6, 7, 8, 9])]

        mean_summer = summer["mean_temp"].mean()
        total_precip_gs = growing["total_precip"].sum(min_count=1)

        FF_COVERAGE_THRESHOLD = 100

        ff_observed = ff_win["min_temp"].notna().sum()
        if ff_observed < FF_COVERAGE_THRESHOLD:
            frost_free = float("nan")
            print(f"WARNING: year {int(yr)} has only {ff_observed} days with min_temp "
                  f"in May-Sep (threshold: {FF_COVERAGE_THRESHOLD}). "
                  f"frost_free_days set to NaN for imputation.")
        else:
            frost_free = int((ff_win["min_temp"] > 0).sum())

        spring_mean= spring["mean_temp"].mean()

        records.append({
            "Year":int(yr),
            "mean_summer_temp_C":round(mean_summer,3),
            "total_growing_precip_mm":round(total_precip_gs,3),
            "frost_free_days":frost_free,  
            "spring_mean_temp_C":round(spring_mean,3),
        })

    df_annual = pd.DataFrame(records).sort_values("Year").reset_index(drop=True)

    # Compute within window anomaly: deviation from 2010 to 2019 spring mean
    baseline_spring = df_annual["spring_mean_temp_C"].mean()
    df_annual["spring_temp_anomaly_C"] = (
        df_annual["spring_mean_temp_C"] - baseline_spring
    ).round(3)


    # Save full series 
    eda_path = os.path.join(OUT_DIR, "climate_annual_full.csv")
    df_annual.to_csv(eda_path, index=False)
    print(f"\nSaved full annual series: {eda_path} ({len(df_annual)} rows)")

    # Filter to modeling window 
    df_model = df_annual[
        (df_annual["Year"] >= 2010) & (df_annual["Year"] <= 2019)
    ][["Year", "mean_summer_temp_C", "total_growing_precip_mm",
       "frost_free_days", "spring_temp_anomaly_C"]].copy()

    model_path = os.path.join(OUT_DIR, "climate_2010_2019.csv")
    df_model.to_csv(model_path, index=False)
    print(f"\nSaved modeling window: {model_path}")
    print(f"\nModeling window (2010 to 2019):\n{df_model.to_string(index=False)}")

    return df_model

### Merge_ABC()

Merges the 3 datasets we will be using and imputes missing values (only on Dataset_C).

For Imputation missing climate values were filled with the column mean across the 2010-2019 window before merging to preserve all 10 years.

* Imputed values are displayed for transparency.

In [ ]:
def merge_ABC():
    """
    Load the three modeling-window CSVs, impute missing climate values,
    and save two merged outputs: one for EDA and one for modeling.

    Merge strategy:
        Inner join on Year across all three datasets.

    Returns
    -------
    df_model : pd.DataFrame
        Modeling-ready merged dataset.
    """
    os.makedirs(OUT_DIR, exist_ok=True)

    # Load modeling files
    df_b = pd.read_csv(os.path.join(OUT_DIR, "wheat_yield_2010_2019.csv"))
    df_a = pd.read_csv(os.path.join(OUT_DIR, "bryophyte_annual_features.csv"))
    df_c = pd.read_csv(os.path.join(OUT_DIR, "climate_2010_2019.csv"))

    print(f"Dataset B shape: {df_b.shape}")
    print(f"Dataset A shape: {df_a.shape}")
    print(f"Dataset C shape: {df_c.shape}")

    # Impute missing climate values with column mean
    climate_feature_cols = [
        "mean_summer_temp_C", "total_growing_precip_mm",
        "frost_free_days", "spring_temp_anomaly_C"
    ]

    nan_counts = df_c[climate_feature_cols].isna().sum()
    if nan_counts.any():
        for col in climate_feature_cols:
            missing_mask = df_c[col].isna()
            if missing_mask.any():
                col_mean = df_c[col].mean()
                df_c.loc[missing_mask, col] = round(col_mean, 3)


    # Merge all three on Year (inner join)
    df_eda = df_b.merge(df_a, on="Year", how="inner") \
                 .merge(df_c, on="Year", how="inner") \
                 .sort_values("Year").reset_index(drop=True)


    # EDA merged- all columns
    eda_path = os.path.join(OUT_DIR, "EDA_merged_ABC_2010_2019.csv")
    df_eda.to_csv(eda_path, index=False)
    print(f"\nSaved EDA merged: {eda_path}")

    # Model merged- model-input columns only
    model_cols = [
        "Year",
        "wheat_yield_bu_acre",
        "richness_per_site", "ids_per_site",
        "mean_summer_temp_C", "total_growing_precip_mm",
        "frost_free_days", "spring_temp_anomaly_C"
    ]
    df_model = df_eda[model_cols].copy()

    model_path = os.path.join(OUT_DIR, "Model_merged_ABC_2010_2019.csv")
    df_model.to_csv(model_path, index=False)
    print(f"\nSaved model merged: {model_path}")

    return df_model

In [92]:

if __name__ == "__main__":

    df_A = preprocess_A()
    print("\npreprocess_A complete.")

    df_B = preprocess_B()
    print("\npreprocess_B complete.")
    
    # ONLY RUN ONCE TO GET CLIMATE FILES
    # fetch_C()
    
    df_C = preprocess_C()
    print("\npreprocess_C complete.")

    df_ABC = merge_ABC()
    print("\nmerge_ABC() complete.")


T01A raw shape: (5400, 14)
T19B raw shape: (24422, 15)
T01A after cleaning: (458, 4)

Saved merged cleaned data: preprocessing_results\merged_bryophyte_cleaned.csv
Saved annual features: preprocessing_results\bryophyte_annual_features.csv

preprocess_A complete.
Raw shape: (117, 2)

Saved full historical series: preprocessing_results\wheat_yield_full_historical.csv  (114 years)

Saved modeling window: preprocessing_results\wheat_yield_2010_2019.csv

preprocess_B complete.

Combined raw shape: (3652, 31)

Saved full annual series: preprocessing_results\climate_annual_full.csv (10 rows)

Saved modeling window: preprocessing_results\climate_2010_2019.csv

Modeling window (2010 to 2019):
 Year  mean_summer_temp_C  total_growing_precip_mm  frost_free_days  spring_temp_anomaly_C
 2010              14.391                    424.4            136.0                 -0.846
 2011              14.851                    353.0            143.0                 -1.386
 2012              16.327         